# 02 — Baseline ML: previsione ON/OFF dei generatori

**Scopo:** leggere `feature_table_long.parquet` (creato dal notebook 01), addestrare un primo modello **XGBoost** e valutare quanto bene predice lo stato ON/OFF (`target`). Alla fine riconvertiamo **le probabilità predette per i casi di test** in una tabella *wide* con le 2.352 colonne `result_committed_...` nella cartella data/processed

Questo è un **esperimento iniziale**, non ancora una valutazione definitiva per Kaggle. Lo split è cronologico all'80%/20% per **caso di simulazione** (non per singola riga): tutte le 14 × 168 = 2.352 righe di un caso restano nello stesso insieme.


> ⚠️ **Limite della baseline:** le finestre settimanali consecutive possono sovrapporsi tra train e test; inoltre alcune feature derivano da valori idrici relativi a ore future. Per una valutazione rigorosa controlleremo la disponibilità delle feature all'inferenza e introdurremo uno split temporale con intervallo di sicurezza (*gap/embargo*). Per ora verifichiamo l'intera pipeline.

## 1 — Importazioni e percorso del Parquet

Il notebook cerca la cartella della repository partendo dalla directory di lavoro attuale. Funziona se Jupyter parte da `ml-project-ntnu` o da `src ghigu`.

In [ ]:
from pathlib import Path
import json
import time
import time
from tqdm.auto import tqdm
from xgboost.callback import TrainingCallback
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

#questa serve solo per un output figo 
class ProgressCallback(TrainingCallback):
    def __init__(self, total):
        self.pbar = tqdm(total=total, desc="Training XGBoost", unit="albero")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        self.pbar.close()
        return model



REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "sintef_project" / "data" / "processed").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Cartella sintef_project/data/processed non trovata. "
        "Avvia Jupyter dalla repository ml-project-ntnu o dalla cartella src ghigu."
    )

PROCESSED_DIR = REPO_ROOT / "sintef_project" / "data" / "processed"
DATASET_PATH = PROCESSED_DIR / "feature_table_long.parquet"
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Dataset non trovato: {DATASET_PATH}. Esegui prima il notebook 01.")

print("Dataset:", DATASET_PATH)

## 2 — Selezione delle feature e caricamento dati

**Le feature sono gli input `X`; `target` è l'etichetta `y`.**

Non usiamo `run_no` (identificativo del caso), `starttime` o `timestamp` come feature: i primi due servono a dividere/ricostruire i dati; il timestamp assoluto per ora non serve. **Conserviamo `hour`**, perché l'ora all'interno della finestra di 168 ore potrebbe essere informativa.

Usiamo `generator` e `plant` come variabili categoriche; XGBoost le gestisce nativamente con `enable_categorical=True`. Sono incluse le **sette feature topologiche** del notebook 01, più le altre caratteristiche numeriche e i vincoli idrici.

In [ ]:
FEATURE_COLUMNS = [
    # Posizione all'interno della settimana e identificazione del generatore
    "hour", "generator", "plant",
    # Prezzo dell'energia
    "price_current",
    # Caratteristiche tecniche del generatore
    "p_min", "p_max", "p_nom", "start_cost", "stop_cost", "penstock",
    # Feature topologiche costruite nel notebook 01
    "direct_total_volume", "direct_mean_fill_ratio", "direct_total_inflow",
    "direct_mean_water_value", "upstream_total_volume",
    "upstream_total_inflow", "upstream_mean_water_value",
    # Vincoli idrici e inflow esterno
    "min_volume_Totak", "min_volume_Byrtevatn", "min_volume_Staavatn",
    "min_flow_b_Kjelavatn", "min_flow_b_Hyljelihyl", "min_flow_b_Vinjevatn",
    "min_flow_b_Byrtevatn", "min_flow_b_Bandak",
    "min_flow_r_Vest_Vassdraget", "inflow_r_Vest_Vassdraget",
]
ID_COLUMNS = ["run_no", "starttime", "generator", "hour"]


# Ok, abbiamo scritto manualmente le colonne che vogliamo leggere dal file .parquet
# e le montiamo insieme in Read_columns. L'idea è di caricare in memoria i dati, ma avendo una divisione 
# tra le colonne che useremo come feature e le colonne con gli id per ricostruire la tabella


READ_COLUMNS = list(dict.fromkeys([*ID_COLUMNS, *FEATURE_COLUMNS, "target"]))
# Read_colums è semplicemente l'unione di tutte le colonne 


# Carichiamo soltanto le colonne che useremo, evitando timestamp.
# praticamente leggiamo dal file indicato da dataset:path, le colonne elencate in Read_Columns
df = pd.read_parquet(DATASET_PATH, columns=READ_COLUMNS)

#df è il datafram con cui lavora pandas, qua diciamo a pandas di trattare starttime come data e ora usando fuso orario UTC
# lo facciamo così possiamo dividere in dati in 80% e 20% in futuro 
df["starttime"] = pd.to_datetime(df["starttime"], utc=True)

# due colonne contengono stringhe e non numeri ( generator e plant ), convertiamo la colonna nel tipo 
# category di Pandas
for column in ["generator", "plant"]:
    df[column] = df[column].astype("category")

# queste sono solo righe di controllo 
assert not df.duplicated(["run_no", "generator", "hour"]).any(), "Chiavi long duplicate"
assert df["target"].notna().all() and df["target"].isin([0, 1]).all(), "Target non binario"
assert len(FEATURE_COLUMNS) == 27

# queste sono solo stampe 
print(f"Dimensioni: {len(df):,} righe × {df.shape[1]} colonne lette")
print(f"Feature utilizzate: {len(FEATURE_COLUMNS)}")
print(f"Casi: {df['run_no'].nunique():,} | Generatori: {df['generator'].nunique()}")
print(f"Target ON: {df['target'].mean():.2%}")
#stampa le prime cinque righe del dataframe per vedere se sono state caricate correttamente
print(df.head(5))

## 3 — Split cronologico: primi 80% dei casi train, ultimi 20% test

Le righe sono raggruppate per `run_no`. Ordiniamo i **casi** per `starttime`, non dividiamo direttamente le ~6,9 milioni di righe: in questo modo tutte le ore e tutti i generatori di una simulazione rimangono insieme. Non c'è mescolamento casuale.
significa che il primo 80% lo prendiamo di dati consecutivi in ordine cronologico.
Ricordiamo che ogni run_no rappresenta una simulazione di 168 ore (7 giorni), relativa a tutti i 14 generatori.

La divisione non elimina ancora la sovrapposizione temporale tra finestre vicine al confine, che miglioreremo successivamente.
Per sovrapposizione si intende il fatto che ogni riga copre una settimana, quindi se uso riga uno come train che copre i giorni 1-7 e come test la riga 2 che copre 2-8 hanno 6 giorni in comune e probabilmente condividono informazioni uguali e diventa inutile 

In [ ]:


#Questo codice serve a creare una tabella contenente tutti i casi di simulazione, ordinarli 
#cronologicamente e calcolare quanti casi destinare al training (80%).


# seleziona solo due colonne dal datafram df, ovvero l'identificativo e lo starttime
#Ricorda che nel nostro dataset long ogni caso occupa 2.352 righe. Quindi, inizialmente, 
#lo stesso run_no e lo stesso starttime compariranno 2.352 volte.
#In questo modo, anziché avere 6.872.544 righe, otteniamo 2.922 righe, una per ogni caso.
cases = (
    df[["run_no", "starttime"]]
    .drop_duplicates()
    .sort_values(["starttime", "run_no"])
    .reset_index(drop=True)
)
#Risultato
#La variabile cases sarà un piccolo DataFrame simile a questo:
#Indice	run_no	starttime
#0	1	2015-01-01
#1	2	2015-01-02
#2	3	2015-01-03
#...	...	...
#2921	2922	2022-12-31

n_train_cases = int(len(cases) * 0.80)
# 2922 *  0.80 = 2337.6

#2.337 casi per il training
#585 casi per il test

# controllo inutile 
if n_train_cases == 0 or n_train_cases == len(cases):
    raise ValueError("Numero di casi insufficiente per lo split train/test")

#Queste tre righe servono a identificare quali casi appartengono al training 
#quali al test, verificando che nessun caso finisca in entrambi.
#Partiamo dall'interno:
# cases.iloc[:n_train_cases] seleziona i primi 80% dei casi, che abbiamo precedentemente ordinato cronologicamente.
# ["run_no"] estrae soltanto gli identificativi di quei casi.
# set(...) trasforma gli identificativi in un insieme Python, che contiene elementi unici, senza duplicati.

train_ids = set(cases.iloc[:n_train_cases]["run_no"]) #sono vettori che contengono i run_no id 
test_ids = set(cases.iloc[n_train_cases:]["run_no"])
assert train_ids.isdisjoint(test_ids)

train_mask = df["run_no"].isin(train_ids)
test_mask = ~train_mask
#il risultato è una mschera booleana 
#run_no	train_mask	test_mask
#1	True	False
#1	True	False
#2	True	False
#4	False	True
#4	False	True


#df.loc permette di selezionare righe e colonne usanto la sintassi df.loc[righe, colonne], righe espresse tramite maschera boolena
X_train = df.loc[train_mask, FEATURE_COLUMNS].copy() #.copy() Crea una copia indipendente dei dati selezionati. Questo è utile perché successivamente potremmo modificare X_train (per esempio trasformare alcune feature) senza modificare accidentalmente il DataFrame originale df
#stesse righe del training ma con la colonna target 
y_train = df.loc[train_mask, "target"].to_numpy(dtype="uint8") 

X_test = df.loc[test_mask, FEATURE_COLUMNS].copy()
y_test = df.loc[test_mask, "target"].to_numpy(dtype="uint8")

'''Queste due istruzioni servono a conservare le informazioni necessarie per ricostruire le 
predizioni nel formato wide originale, dopo che il modello XGBoost avrà effettuato le previsioni.
Ricorda che il modello riceverà soltanto le feature (X_test) e produrrà un array di probabilità. 
Dovremo poi capire a quale caso, generatore e ora appartiene ciascuna probabilità.

Supponiamo che XGBoost produca: predictions = [0.92, 0.81, 0.15, 0.73, ...], con test_keys e l'altro lo possiamo riassociare
alle righe originali 

'''
test_keys = df.loc[test_mask, ID_COLUMNS].copy().reset_index(drop=True)
# Identificativi esclusivamente per ricostruire le predizioni wide.
#seleziona dal DataFrame originale:
#- Le righe del test, tramite test_mask.
#- Le quattro colonne identificative, tramite ID_COLUMNS.

generator_order = list(pd.unique(df["generator"]))
# Manteniamo l'ordine di prima comparsa dei generatori, come nel long originario.


if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
    raise ValueError("Train o test contiene una sola classe: ROC-AUC non calcolabile")

print(f"Casi train: {len(train_ids):,} | righe train: {len(X_train):,}")
print(f"Casi test:  {len(test_ids):,} | righe test:  {len(X_test):,}")
print("Periodo train:", cases.iloc[0]["starttime"], "→", cases.iloc[n_train_cases - 1]["starttime"])
print("Periodo test: ", cases.iloc[n_train_cases]["starttime"], "→", cases.iloc[-1]["starttime"])
print(f"% ON train: {y_train.mean():.2%} | % ON test: {y_test.mean():.2%}")

## 4 — Addestramento del modello XGBoost

**XGBoost** combina molti alberi decisionali in sequenza (*gradient boosting*). È una buona baseline per dati tabellari numerici e categorici, senza richiedere normalizzazione delle feature. Partiamo con iperparametri ragionevoli, **senza tuning**.

Per lo scopo di questo notebook **il test non viene utilizzato per ottimizzare il modello**: misuriamo il risultato solo dopo l'addestramento. Sull'intero dataset il training può impiegare alcuni minuti o più, a seconda del computer.


credo che in futuro quando il modello sarà abbastanza preciso e ci piace useremo anche il test per addestrare ? 




Iperparametro	    Cosa controlla

n_estimators	    Numero di alberi
max_depth	        Complessità di ciascun albero
learning_rate	    Intensità del contributo dei nuovi alberi
subsample	        Percentuale di righe campionate per albero
colsample_bytree	Percentuale di feature campionate per albero

Questi saranno tra i principali candidati per il futuro hyperparameter tuning.




In [ ]:
# la classe XGBClassifier crea un modello XGBoost per problemi di classificazione.

model = XGBClassifier(
    objective="binary:logistic",        # il modello deve predirre 1 o 0 ( che a noi significano ON e OFF )
    n_estimators=200,                   # numero di alberi decisionali che XGBoost userà 
    max_depth=5,                        # profondità massima di ciascun albero 
    learning_rate=0.08,                 # uanto contribuisce ogni nuovo albero alla predizione complessiva.
    subsample=0.85,                     # Ogni albero utilizza un campione casuale dell'85% delle righe di training.
    colsample_bytree=0.85,              # È simile a subsample, ma riguarda le colonne anziché le righe.
    tree_method="hist",                 # Indica l'algoritmo utilizzato da XGBoost per costruire gli alberi.
    enable_categorical=True,            # enable_categorical=True
    eval_metric="logloss",              # Indica una metrica utilizzabile da XGBoost per valutare l'errore nelle probabilità predette. La logarithmic loss penalizza soprattutto le predizioni molto sicure ma sbagliate.
    n_jobs=8,                           # Indica quanti thread CPU XGBoost può utilizzare per parallelizzare il lavoro.  
    random_state=42,                    # mposta un seme per le operazioni casuali del modello.
)



# solo per output figo ignorare
model.set_params(
    callbacks=[ProgressCallback(model.n_estimators)]
)

# questa restituisci il valore di un contatore ad alta risoluzione, utile per misurare la durata di una operazione
start = time.perf_counter()

#questa esegue effettivamente l'addestramento
model.fit(X_train, y_train, verbose=False)

#ti stampa quanto ci mette a fare l'addestramento
training_seconds = time.perf_counter() - start
print(f"Addestramento completato in {training_seconds:.1f} secondi")

## 5 — Predizioni e valutazione sul 20% finale

Per ogni riga otteniamo una **probabilità di ON** (`predict_proba`), non soltanto una classe 0/1. Calcoliamo la **ROC-AUC micro** come richiesto dal progetto: poiché i target long sono già tutte le coppie generatore/ora, basta appiattire tutte le righe e calcolare `roc_auc_score(y_test, proba)`.

Mostriamo anche accuracy, precision, recall e F1 usando una soglia indicativa di `0.5`. **La ROC-AUC non richiede alcuna soglia** ed è la metrica principale.




Con il professore e nella competizione Kaggle, la metrica principale è ROC-AUC micro, quindi confrontiamo le probabilità predette dal modello con i valori reali 0 e 1.
In particolare:
- y_test contiene i valori reali: 0 oppure 1.
- probabilities contiene le probabilità di accensione previste da XGBoost, comprese tra 0 e 1.
- predicted_labels contiene le predizioni binarie ottenute applicando la soglia 0.5.


Per esempio:
Target reale	Probabilità ON	Predizione binaria
1	            0.92	        1
0	            0.15	        0
1	            0.65	        1              
0	            0.70	        1
Per ROC-AUC utilizziamo la seconda colonna, confrontandola con la prima. Non utilizziamo la terza.
La ROC-AUC valuta quanto bene il modello assegna probabilità più alte agli esempi realmente ON rispetto a quelli OFF. Non richiede una soglia di 0.5.

In [ ]:
# allora, la funzione predic_proba() pinza in ingresso X_test, 
# cioè le 27 feature del test e restituisce le probabilità associate alle 
# due classi ON e OFF.
# es 
#[
#    [0.10, 0.90],
#    [0.75, 0.25],
#    [0.40, 0.60]
#]
# con la riga sotto prendiamo la seconda colonna di queste prob ( la probabilità che sia ON ) e 

probabilities = model.predict_proba(X_test)[:, 1]

# la convertiamo in un 1 o 0 
predicted_labels = (probabilities >= 0.5).astype("uint8")

metrics = {
    #metrica usata dal progetto: roc_auc_micro
    "roc_auc_micro": float(roc_auc_score(y_test, probabilities)),
    #accuracy ( percentuale di quelle indovinate )
    "accuracy_at_0_5": float(accuracy_score(y_test, predicted_labels)),
    #precisione ( tra tutti i generatori che il modello ha previsto ON, quanti erano davvero ON)
    "precision_at_0_5": float(precision_score(y_test, predicted_labels, zero_division=0)),
    # Tra tutti i generatori che erano realmente ON, quanti ne abbiamo identificati correttamente?
    "recall_at_0_5": float(recall_score(y_test, predicted_labels, zero_division=0)),
    # L'F1-score combina precision e recall in un'unica misura, utilizzando la media armonica:
    "f1_at_0_5": float(f1_score(y_test, predicted_labels, zero_division=0)),
    
    "n_train_rows": int(len(y_train)),
    "n_test_rows": int(len(y_test)),
    "n_train_cases": int(len(train_ids)),
    "n_test_cases": int(len(test_ids)),
    "training_seconds": round(training_seconds, 2),
}

for name, value in metrics.items():
    print(f"{name}: {value}")
print("\nMatrice di confusione [righe = vero, colonne = predetto]:")
print(confusion_matrix(y_test, predicted_labels, labels=[0, 1]))

print("\nFeature più importanti (importanza XGBoost relativa):")
importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print(importance.head(15).rename("importance").to_frame())

## 6 — Riconversione long → wide delle **probabilità** sul test

Non abbiamo bisogno di importare il notebook 01: qui inseriamo una piccola funzione di conversione autonoma, equivalente alla sua `to_wide()`. In generale, quando più notebook devono usare la stessa funzione, è meglio spostarla in un **modulo `.py` condiviso** (per esempio `src/utils.py`) e importarla, invece di importare direttamente un `.ipynb`.

Il risultato ha **una riga per caso** e **2.352 colonne di probabilità** `result_committed_<generator>_t<ora>` in ordine generatore/ora. Non ricostruiamo *target* binari, perché per ROC-AUC e classificazione probabilistica è utile mantenere i punteggi tra 0 e 1.

`total_calculation_time` **non era nel long** e non si può prevedere: viene aggiunta una colonna vuota (`NaN`) soltanto per riprodurre la struttura del CSV originale. **Questo file è una verifica sul set di test interno, non ancora una submission Kaggle**: per quella dovremo verificare `sample_submission.csv` e `prediction_mapping.csv`.

In [ ]:
def predictions_to_wide(
    long_predictions: pd.DataFrame,
    generators: list[str],
    hours: int = 168,
) -> pd.DataFrame:
    """Da (run_no, starttime, generator, hour, prediction) alle colonne wide originali."""
    required = ["run_no", "starttime", "generator", "hour", "prediction"]
    if long_predictions[required].isna().any().any():
        raise ValueError("Chiavi o predizioni mancanti")
    if long_predictions.duplicated(["run_no", "generator", "hour"]).any():
        raise ValueError("Duplicati per (run_no, generator, hour)")
    if not long_predictions["prediction"].between(0, 1).all():
        raise ValueError("Le predizioni devono essere probabilità tra 0 e 1")

    small = long_predictions[required].copy()
    small["target_column"] = (
        "result_committed_" + small["generator"].astype(str)
        + "_t" + small["hour"].astype(str)
    )
    target_columns = [
        f"result_committed_{generator}_t{hour}"
        for generator in generators for hour in range(hours)
    ]

    wide = small.pivot(index=["run_no", "starttime"], columns="target_column", values="prediction")
    missing = set(target_columns) - set(wide.columns)
    if missing:
        raise ValueError(f"Mancano {len(missing)} colonne generatore/ora: {sorted(missing)[:5]}")
    wide = wide.reindex(columns=target_columns).reset_index()
    if wide[target_columns].isna().any().any():
        raise ValueError("Alcuni casi non hanno tutte le 168 ore × tutti i generatori")

    wide = wide.rename(columns={"run_no": "Run No"})
    wide["total_calculation_time"] = np.nan  # Solo placeholder: non disponibile nel long
    return wide[["Run No", "starttime", "total_calculation_time", *target_columns]]

# Associa a ciascuna riga di test la probabilità predetta.
long_predictions = test_keys.assign(prediction=probabilities.astype("float32"))
wide_predictions = predictions_to_wide(long_predictions, generator_order)
assert len(wide_predictions) == len(test_ids)
assert wide_predictions.shape[1] == 2355  # 3 metadata + 14 * 168 target

print(f"Tabella wide: {wide_predictions.shape[0]:,} casi × {wide_predictions.shape[1]:,} colonne")
print("Prime colonne:")
display(wide_predictions.iloc[:3, :9])

## 7 — Salvataggio del modello e dei risultati

Salviamo tutto in `sintef_project/data/processed/`:

- `baseline_xgboost.json`: il modello addestrato, riutilizzabile senza rifare il training.
- `baseline_validation_predictions_wide.csv`: probabilità per il **20% di casi trattenuti come test**, in formato wide.
- `baseline_metrics.json`: metriche e dimensioni del test.

Il dataset Parquet iniziale **non viene modificato**.

In [ ]:
MODEL_PATH = PROCESSED_DIR / "baseline_xgboost.json"
WIDE_PATH = PROCESSED_DIR / "baseline_validation_predictions_wide.csv"
METRICS_PATH = PROCESSED_DIR / "baseline_metrics.json"

model.save_model(MODEL_PATH)
wide_predictions.to_csv(WIDE_PATH, index=False)
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("Modello:", MODEL_PATH)
print("Predizioni wide:", WIDE_PATH)
print("Metriche:", METRICS_PATH)

### Come leggere il risultato

La **ROC-AUC** misura se il modello assegna in generale probabilità maggiori ai casi ON rispetto agli OFF: `0.5` corrisponde a una classificazione casuale, `1.0` a una separazione perfetta. Non è una percentuale di predizioni corrette (quella è l'accuracy).

Anche con una ROC-AUC elevata, questo esperimento **non dimostra ancora** che il modello generalizzi a dati futuri: le finestre possono sovrapporsi, le feature devono essere effettivamente disponibili al momento della previsione e serviranno validazione temporale più rigorosa e confronto con baseline semplici.